# ProxyTrace 06. 사실 판별 에이전트

05에서 순위를 매긴 주장을 하나씩 들고, **에이전트가 근거 자료를 직접 찾아와 사실인지 판정**해요.

```
주장 → GPT: "어디서 확인하지?" → 도구 호출 → 결과 읽기 → (부족하면 다른 도구) → 판정 제출
                                                                         ↓
                                         코드 검사관: 근거가 진짜 그 자료에 있나? 기준일 이전 자료인가?
```

| 역할 | 누가 |
| --- | --- |
| 생각하고 판정 (두뇌) | GPT |
| 자료 가져오기 (손발) | 도구 = 우리가 만든 파이썬 함수 |
| 판정 재확인 (검사관) | 코드 |

**도구 (이번 버전)**: `search_proxy` 위임장 검색 · `edgar_search` SEC 공시 찾기 · `read_filing` 공시 읽기 · `calc` 계산 · `submit_verdict` 판정 제출
→ 웹페이지 스냅샷(11번 안건), H-1B 데이터(12번 안건)는 다음에 추가해요.

결과: `data/verdicts/{TICKER}/{제출일}_verdicts.json`, `.csv`

In [ ]:
%pip install openai python-dotenv requests beautifulsoup4 lxml pandas -q

## 1. 설정
- `.env`에 `OPENAI_API_KEY`와 `SEC_USER_AGENT`가 둘 다 있어야 해요.
- `AS_OF` = 위임장이 나온 날. **이 날짜 이후 자료는 근거로 못 써요.** 주총 결과처럼 미래 정보를 보고 판정하면 공정하지 않으니까요.
- 처음엔 4번 안건, 주장 3개만 시험해요.

In [ ]:
import ast
import csv
import json
import os
import re
import time
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import warnings
from bs4 import XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

load_dotenv(".env")   # OPENAI_API_KEY, SEC_USER_AGENT

# ▶ 01~05와 같은 티커
TICKER = "META"

# ▶ 판별은 추출보다 어려워서 큰 모델을 써요. 비용이 걱정되면 "gpt-4.1-mini"로 시작해도 돼요.
MODEL = "gpt-4o"

# ▶ 처음엔 4번 안건, 주장 3개만 시험 → 괜찮으면 늘리기 (None = 전부)
TEST_PROPOSALS = [3]
MAX_CLAIMS = 5
MAX_RETRY = 1         # 검사기가 판정을 막았을 때 다시 찾아보게 하는 횟수
MAX_STEPS = 6          # 주장 하나당 GPT와 주고받는 최대 횟수 (마지막 차례엔 판정을 강제해요)

client = OpenAI()
HEADERS = {"User-Agent": os.environ["SEC_USER_AGENT"]}   # .env에 SEC_USER_AGENT="이름 이메일"

T = TICKER.lower()
manifest = json.loads((Path("data/raw") / T / "manifest.json").read_text(encoding="utf-8"))
CIK = manifest["cik"]
FILING = manifest["proxies"][0]["filingDate"]
AS_OF = FILING          # ★ 이 날짜 이후 자료는 근거로 쓰지 않아요 (위임장이 나온 날)
PROXY_URL = manifest["proxies"][0]["url"]

pages = json.loads((Path("data/clean") / T / f"{FILING}_pages.json").read_text(encoding="utf-8"))["pages"]
claim_file = Path("data/claims") / T / f"{FILING}_claims_esg.json"
claims = json.loads(claim_file.read_text(encoding="utf-8"))["claims"]

# 안건별 주주 글 + 이사회 답변 (검사기가 "같은 안건 글을 근거로 썼나" 확인할 때 써요)
_prop = json.loads((Path("data/parsed") / T / f"{FILING}_proposals.json").read_text(encoding="utf-8"))
_prop = _prop["proposals"] if isinstance(_prop, dict) else _prop
PROP_TEXT = {p["number"]: p.get("proponent_text", "") + " " + p.get("board_text", "") for p in _prop}
PROP_PAGES = {p["number"]: p.get("pages") for p in _prop}   # 안건이 실린 쪽 범위 (예: [67, 70])

OUT_DIR = Path("data/verdicts") / T
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE = Path("data/cache") / T          # SEC에서 받은 문서를 다시 안 받도록 저장
CACHE.mkdir(parents=True, exist_ok=True)

print(f"{manifest['company']} 위임장 {FILING} | 기준일 {AS_OF} | 주장 {len(claims)}개")

## 2. SEC 공시 받기
01·02 노트북 코드를 재사용해요. 한 번 받은 공시는 `data/cache/`에 저장해서 다시 안 받아요.

In [ ]:
# ── SEC 공시 받기 (01 노트북 코드 재사용) ─────────────────────
def sec_get(url: str) -> requests.Response:
    r = requests.get(url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    time.sleep(0.2)          # SEC 제한: 초당 10회
    return r


def all_filings(cik: str) -> list[dict]:
    """회사의 공시 목록 전체. 8-K는 items(예: '5.07')도 같이 가져와요."""
    sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json").json()

    def rows(block):
        keys = ["form", "filingDate", "accessionNumber", "primaryDocument", "items"]
        return [dict(zip(keys, v)) for v in zip(*(block.get(k, [""] * len(block["form"])) for k in keys))]

    out = rows(sub["filings"]["recent"])
    for f in sub["filings"].get("files", []):
        out += rows(sec_get(f"https://data.sec.gov/submissions/{f['name']}").json())
    for r in out:
        acc = r["accessionNumber"].replace("-", "")
        r["url"] = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{r['primaryDocument']}"
    return out


FILINGS = all_filings(CIK)
URL_DATE = {r["url"]: r["filingDate"] for r in FILINGS}
URL_DATE[PROXY_URL] = FILING
print(f"공시 목록 {len(FILINGS)}건")


def html_to_text(html: str) -> str:
    """02 노트북과 같은 방식: 숨김 영역 지우고, 문단 단위로 줄바꿈"""
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style", "head"]):
        t.decompose()
    for t in soup.find_all(re.compile(r"^ix:header$", re.I)):
        t.decompose()
    for t in soup.find_all(style=re.compile(r"display\s*:\s*none", re.I)):
        t.decompose()
    for t in soup.find_all(["td", "th"]):
        t.append(" | ")
    for t in soup.find_all(["p", "div", "tr", "li", "h1", "h2", "h3", "h4", "table", "br"]):
        t.append("\n")
    text = soup.get_text("").replace("\xa0", " ")
    lines = [re.sub(r"[ \t]+", " ", ln).strip(" |") for ln in text.split("\n")]
    return "\n".join(ln for ln in lines if ln)


DOC_TEXT = {}   # url → 정리된 전체 텍스트 (검사관이 근거를 대조할 때 써요)


def load_filing(url: str) -> str:
    if url in DOC_TEXT:
        return DOC_TEXT[url]
    path = CACHE / (re.sub(r"[^A-Za-z0-9]+", "_", url)[-120:] + ".txt")
    if path.exists():
        text = path.read_text(encoding="utf-8")
    else:
        text = html_to_text(sec_get(url).text)
        path.write_text(text, encoding="utf-8")
    DOC_TEXT[url] = text
    return text


# 우리 위임장도 같은 방식으로 등록 (쪽 번호가 있는 02 결과 사용)
DOC_TEXT[PROXY_URL] = "\n".join(p["text"] for p in pages)

## 3. 도구
GPT는 도구를 직접 실행하지 못해요. **"이 도구를 이 값으로 실행해 줘"라고 요청만** 하고, 실행은 우리 코드가 해요.
`TOOLS`는 GPT에게 주는 **도구 설명서**예요. GPT는 이 설명을 읽고 언제 어떤 도구를 쓸지 정해요.

In [ ]:
# ── 도구 (에이전트의 손발) ─────────────────────────────────
def ws(s: str) -> str:
    s = s.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", s).strip().lower()


def passages(text: str, query: str, n: int = 2, width: int = 600) -> list[str]:
    """검색어 단어가 많이 모여 있는 부분을 n개 골라 앞뒤로 잘라 줘요 (10-K처럼 긴 문서를 통째로 주지 않으려고)"""
    terms = [t for t in re.findall(r"[a-z0-9%\-]+", query.lower()) if len(t) > 2]
    low = text.lower()
    hits = sorted({m.start() for t in terms for m in re.finditer(re.escape(t), low)})
    scored = []
    for h in hits:
        win = low[max(0, h - width // 2): h + width // 2]
        scored.append((sum(t in win for t in terms), h))
    out, used = [], []
    for score, h in sorted(scored, reverse=True):
        if any(abs(h - u) < width for u in used):
            continue
        used.append(h)
        out.append(text[max(0, h - width // 2): h + width // 2].replace("\n", " "))
        if len(out) == n:
            break
    return out


def search_proxy(query: str) -> str:
    """우리 위임장(2026)에서 검색. 쪽 번호와 함께 돌려줘요."""
    terms = [t for t in re.findall(r"[a-z0-9%\-]+", query.lower()) if len(t) > 2]
    ranked = sorted(pages, key=lambda p: -sum(p["text"].lower().count(t) for t in terms))
    out = []
    for p in ranked[:3]:
        for s in passages(p["text"], query, n=1):
            out.append(f"[proxy p.{p['page_no']}] {s}")
    return json.dumps({"source_url": PROXY_URL, "source_date": FILING, "results": out}, ensure_ascii=False)


def edgar_search(form: str, date_from: str, date_to: str, item: str = "") -> str:
    """SEC 공시 목록에서 서류 종류·기간으로 찾기. 기준일(AS_OF) 이후는 자동으로 빼요."""
    date_to = min(date_to, AS_OF)
    hits = [r for r in FILINGS
            if r["form"] == form and date_from <= r["filingDate"] <= date_to
            and (not item or item in (r.get("items") or ""))]
    hits = sorted(hits, key=lambda r: r["filingDate"], reverse=True)[:10]
    return json.dumps([{k: r[k] for k in ("form", "filingDate", "items", "url")} for r in hits], ensure_ascii=False)


def read_filing(url: str, query: str) -> str:
    """SEC 공시를 받아서 검색어 주변 부분만 돌려줘요."""
    if not url.startswith("https://www.sec.gov/"):
        return json.dumps({"error": "sec.gov 주소만 읽을 수 있어요"})
    date = URL_DATE.get(url, "")
    if date and date > AS_OF:
        return json.dumps({"error": f"기준일 {AS_OF} 이후 자료라 쓸 수 없어요"})
    text = load_filing(url)
    return json.dumps({"source_url": url, "source_date": date, "results": passages(text, query)}, ensure_ascii=False)


def calc(expression: str) -> str:
    """사칙연산만 안전하게 계산 (eval은 위험해서 안 써요)"""
    expression = expression.replace(",", "")   # 1,234 같은 숫자도 계산되게
    allowed = (ast.Expression, ast.BinOp, ast.UnaryOp, ast.Constant, ast.Add, ast.Sub,
               ast.Mult, ast.Div, ast.Pow, ast.USub, ast.UAdd, ast.Mod)
    tree = ast.parse(expression, mode="eval")
    if not all(isinstance(n, allowed) for n in ast.walk(tree)):
        return json.dumps({"error": "숫자와 + - * / ( ) 만 쓸 수 있어요"})
    return json.dumps({"result": eval(compile(tree, "<calc>", "eval"))})


TOOL_FUNCS = {"search_proxy": search_proxy, "edgar_search": edgar_search,
              "read_filing": read_filing, "calc": calc}

VERDICTS = ["확인됨", "부분 확인", "상충", "확인 불가"]

# GPT에게 주는 도구 설명서 — GPT는 이 설명을 읽고 언제 어떤 도구를 쓸지 정해요
TOOLS = [
    {"type": "function", "function": {
        "name": "search_proxy",
        "description": "Search the company's current proxy statement (the document the claim came from). Good for ownership tables, voting power, board facts, things stated elsewhere in the proxy.",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "edgar_search",
        "description": "List the company's SEC filings by form type and filing date range (YYYY-MM-DD). For past annual meeting vote results use form '8-K' with item '5.07', filed within ~2 weeks after the meeting. Other forms: '10-K', 'DEF 14A'.",
        "parameters": {"type": "object", "properties": {
            "form": {"type": "string"}, "date_from": {"type": "string"}, "date_to": {"type": "string"},
            "item": {"type": "string", "description": "8-K item number like '5.07', or empty"}},
            "required": ["form", "date_from", "date_to", "item"]}}},
    {"type": "function", "function": {
        "name": "read_filing",
        "description": "Read an SEC filing by its url (from edgar_search) and return the passages that best match the query.",
        "parameters": {"type": "object", "properties": {"url": {"type": "string"}, "query": {"type": "string"}},
                       "required": ["url", "query"]}}},
    {"type": "function", "function": {
        "name": "calc",
        "description": "Arithmetic on numbers, e.g. '2025 + 3' or '345.2 / 2600.4 * 100'.",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "submit_verdict",
        "description": "Submit the final verdict. Call exactly once when done.",
        "parameters": {"type": "object", "properties": {
            "verdict": {"type": "string", "enum": VERDICTS},
            "evidence_quote": {"type": "string", "description": "Short passage copied EXACTLY from a tool result. Empty if 확인 불가."},
            "source_url": {"type": "string"},
            "source_date": {"type": "string"},
            "explanation_ko": {"type": "string", "description": "판정 이유를 한국어 1~2문장으로"}},
            "required": ["verdict", "evidence_quote", "source_url", "source_date", "explanation_ko"]}}},
]

## 3-1. 도구 하나씩 시험해 보기 ⚠ 에이전트를 돌리기 전에 꼭
도구가 제대로 안 되면 에이전트도 헤매요. 결과가 나오는지 먼저 확인하세요.

In [ ]:
print(search_proxy("total voting power Zuckerberg")[:800])

In [ ]:
# 2025년 주총 결과 8-K 찾기 (Meta 2025 주총은 5월 말)
print(edgar_search("8-K", "2025-05-01", "2025-07-31", "5.07"))

In [ ]:
# 위 결과의 url 하나를 넣어서 읽어 보기
hits = json.loads(edgar_search("8-K", "2025-05-01", "2025-07-31", "5.07"))
if hits:
    print(read_filing(hits[0]["url"], "frequency say on pay three years")[:1500])

## 4. 에이전트 지시문
`3. Where to look first`가 **도메인 지식이 들어가는 자리**예요. "이런 주장은 여기서 확인한다"를 정리해 줄수록 에이전트가 덜 헤매요.

In [ ]:
VERIFIER_SYSTEM = f"""You are a fact-checker helping an institutional investor's proxy-voting analyst.
You get ONE claim from a {TICKER} proxy statement filed {AS_OF}. Decide whether outside evidence supports it.

Rules:
1. Use ONLY what your tools return. Never use your own memory as evidence.
2. Use no source dated after {AS_OF}.
3. Where to look first:
   - Past annual meeting vote results → edgar_search form 8-K, item 5.07, filed in the weeks after that meeting; then read_filing.
   - Voting power, share ownership, board facts → search_proxy ("voting power", "beneficial ownership").
   - Business facts, workforce, risks → edgar_search form 10-K (latest before {AS_OF}); then read_filing.
   - What was or was not on a past ballot / in a past proxy → edgar_search form DEF 14A filed in that year; then read_filing.
4. If the first source does not have it, try a different query or another source. Do not repeat the same call.
5. Numbers: use calc only to compare or compute two real numbers (e.g. "60.8 - 60"). Never call calc on a single number. Small rounding differences ("about 60%" vs 61.2%) → 확인됨; a different period or scope → 부분 확인.
6. Verdicts:
   - 확인됨: the evidence matches the claim.
   - 부분 확인: only part matches, or the period/scope differs.
   - 상충: the evidence says something different.
   - 확인 불가: you could not find evidence (this does NOT mean the claim is false).
7. evidence_quote = ONE continuous passage copied EXACTLY from a tool result in THIS conversation (the code checks it).
   Never join pieces with "...". Leave it empty for 확인 불가.
8. Inference is allowed: if the claim follows directly from a fact you found plus simple arithmetic
   (e.g. "approved every three years in 2025" + calc("2025 + 3") = 2028), judge it 확인됨 and quote the base fact.
9. Be efficient. Plan first. Most claims need 2-4 tool calls. Never repeat the same call.
   If "Sources already used for this proposal" are given, try them first.
10. explanation_ko: say what the evidence shows (numbers, dates, which document). Do not just repeat the claim.
11. The claim's own sentence is NOT evidence. The proponent's statement and the board's response in this proposal
    are what we are checking, so never quote them. Find an independent document (a past proxy, 8-K, 10-K) or another
    part of the proxy written for a different purpose (ownership table, compensation section).
12. "Sources already used" are only hints. You must read them yourself with read_filing before quoting them.
13. A request or opinion cannot be 상충. If the claim mixes a fact with a request, judge only the fact.
14. Finish by calling submit_verdict once.
"""

## 5. 에이전트 반복 + 코드 검사관
- `verify()`: GPT가 `submit_verdict`를 부를 때까지 도구를 쓰게 하는 반복문 (최대 `MAX_STEPS`번)
- `check_verdict()`: GPT가 댄 발췌문이 실제 자료에 없으면, 또는 기준일 이후 자료면 → '확인 불가'로 낮춰요
- `trace`: 에이전트가 한 일을 전부 기록해요. 데모에서 \"에이전트가 스스로 찾아갔다\"를 보여줄 때 써요.

In [ ]:
import re

def ask(messages, force_submit=False):
    extra = {"tool_choice": {"type": "function", "function": {"name": "submit_verdict"}}} if force_submit else {}
    return client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, temperature=0, **extra)


def _words(s: str) -> set:
    return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if len(w) >= 4}


def check_verdict(v: dict, trace: list, claim: dict) -> dict:
    """코드 검사관: GPT가 낸 판정이 근거 규칙을 지켰는지 확인"""
    notes = []
    seen = ws(" ".join(t["result"] for t in trace))
    full = ws(" ".join(DOC_TEXT.get(u, "") for u in {t.get("url") for t in trace} | {v.get("source_url")} if u))
    q = ws(v.get("evidence_quote", ""))
    parts = [p.strip() for p in re.split(r"\.\.\.|…", q) if p.strip()]

    if v["verdict"] != "확인 불가":
        # ① 근거 발췌문이 에이전트가 실제로 본 자료에 있나? (지어낸 근거 차단)
        #    "..."으로 이어 붙였으면 조각마다 따로 확인
        if not parts or any(p not in seen and p not in full for p in parts):
            notes.append("근거 발췌문을 원자료에서 찾지 못해 '확인 불가'로 낮춤")
            v["verdict"] = "확인 불가"
        # ③ 주장 문장 자체, 또는 같은 안건의 주주 글·이사회 답변을 근거로 썼나? (자기 인용 차단)
        #    (주장 문장은 항상 그 안건 글 안에 있으니, 안건 글 검사 하나로 둘 다 잡혀요.
        #     위원회 설명처럼 위임장의 다른 섹션에서 인용한 건 통과해요.)
        elif v.get("source_url") == PROXY_URL:
            own = ws(PROP_TEXT.get(claim["proposal"], ""))
            rng = PROP_PAGES.get(claim["proposal"])
            # 발췌문이 실린 쪽 찾기 (글자 비교가 조금 어긋나도 쪽 번호로 한 번 더 잡아요)
            key = parts[0][:60]
            hit_pages = [p["page_no"] for p in pages if key in ws(p["text"])]
            in_range = bool(rng and hit_pages and all(rng[0] <= n <= rng[1] for n in hit_pages))
            if (own and any(p in own for p in parts)) or in_range:
                notes.append("같은 안건의 주주 글·이사회 답변을 근거로 써서 '확인 불가'로 낮춤")
                v["verdict"] = "확인 불가"
    # ② 기준일 이후 자료인가? (미래 정보 차단)
    if v.get("source_date") and v["source_date"] > AS_OF:
        notes.append(f"기준일 {AS_OF} 이후 자료라 '확인 불가'로 낮춤")
        v["verdict"] = "확인 불가"
    v["check_notes"] = " / ".join(notes)
    return v


def verify(claim: dict, hints: list = None) -> dict:
    messages = [
        {"role": "system", "content": VERIFIER_SYSTEM},
        {"role": "user", "content": (
            f"Claim ({claim['speaker']}, proxy p.{claim['page_no']}, proposal {claim['proposal']}: {claim['proposal_topic']}):\n"
            f"\"{claim['quote']}\"\n(Korean summary: {claim['claim_ko']})\nTime reference: {claim.get('time_ref') or '-'}"
            + (("\nSources already used for this proposal:\n" + "\n".join(hints)) if hints else ""))},
    ]
    done_calls = {}   # 같은 도구를 같은 값으로 또 부르면 다시 실행하지 않아요
    trace, usage = [], {"input": 0, "output": 0}
    retries = 0       # 검사기가 판정을 돌려보낸 횟수

    for step in range(MAX_STEPS):
        last = step == MAX_STEPS - 1
        if last:   # 마지막 차례: 찾은 것까지로 판정을 내게 강제
            messages.append({"role": "user", "content": "This is your last turn. Submit your best verdict now from what you found."})
        resp = ask(messages, force_submit=last)
        usage["input"] += resp.usage.prompt_tokens
        usage["output"] += resp.usage.completion_tokens
        msg = resp.choices[0].message
        messages.append(msg.model_dump(exclude_none=True))

        if not msg.tool_calls:          # 도구도 판정도 없이 말만 하면 다시 재촉
            messages.append({"role": "user", "content": "Use a tool, or call submit_verdict."})
            continue

        final = None
        for call in msg.tool_calls:
            name, args = call.function.name, json.loads(call.function.arguments or "{}")
            if name == "submit_verdict":
                v = check_verdict({**args}, trace, claim)
                # 검사기가 막았고, 아직 기회가 남았으면 → 끝내지 말고 에이전트에게 돌려보내기
                if (args.get("verdict") != "확인 불가" and v["verdict"] == "확인 불가"
                        and retries < MAX_RETRY and not last):
                    retries += 1
                    result = json.dumps({
                        "rejected": v["check_notes"],
                        "what_to_do": ("That quote cannot be used as evidence. Look for an independent source: "
                                       "another section of the proxy (board committees, corporate governance, ownership table), "
                                       "a 10-K or an 8-K. Read it with a tool, then submit again. "
                                       "If no such source exists, submit 확인 불가."),
                    }, ensure_ascii=False)
                else:
                    if retries:
                        v["check_notes"] = " / ".join(x for x in [f"검사기가 {retries}번 돌려보낸 뒤 재제출", v["check_notes"]] if x)
                    final = v
                    result = "received"
            else:
                key = (name, json.dumps(args, sort_keys=True))
                if key in done_calls:
                    result = json.dumps({"note": "You already made this exact call. Use the earlier result or try something different."})
                else:
                    try:
                        result = TOOL_FUNCS[name](**args)
                    except Exception as e:      # 도구가 실패해도 에이전트가 다른 방법을 찾게
                        result = json.dumps({"error": str(e)[:200]})
                    done_calls[key] = result
                trace.append({"step": step + 1, "tool": name, "args": args,
                              "url": args.get("url") or (PROXY_URL if name == "search_proxy" else None),
                              "result": result})
            messages.append({"role": "tool", "tool_call_id": call.id, "content": result})

        if final:
            return {**final, "steps": len(trace), "usage": usage, "trace": trace}

    return {"verdict": "확인 불가", "evidence_quote": "", "source_url": "", "source_date": "",
            "explanation_ko": f"{MAX_STEPS}번 안에 판정하지 못함", "check_notes": "시도 횟수 초과",
            "steps": len(trace), "usage": usage, "trace": trace}

## 6. 실행

In [ ]:
targets = [c for c in claims
           if c.get("checkable") and c.get("priority") != "제외"
           and (TEST_PROPOSALS is None or c["proposal"] in TEST_PROPOSALS)]
order = {"1순위": 0, "2순위": 1, "3순위": 2}
targets = sorted(targets, key=lambda c: (order.get(c.get("priority"), 9), -c.get("priority_score", 0)))
if MAX_CLAIMS:
    targets = targets[:MAX_CLAIMS]
print(f"판별할 주장 {len(targets)}개")

results, total = [], {"input": 0, "output": 0}
for c in targets:
    # 같은 안건에서 앞서 근거로 쓴 문서를 힌트로 넘겨요 (처음부터 다시 찾지 않게)
    hints = sorted({f"- {r['source_url']} ({r['source_date']}) used for: {r['claim_ko']}"
                    for r in results if r["proposal"] == c["proposal"] and r.get("source_url") and r["verdict"] != "확인 불가"})
    v = verify(c, hints)
    total = {k: total[k] + v["usage"][k] for k in total}
    results.append({**c, **{k: v[k] for k in ("verdict", "evidence_quote", "source_url", "source_date",
                                              "explanation_ko", "check_notes", "steps")}, "trace": v["trace"]})
    print(f"{c['claim_id']} → {v['verdict']:6} (도구 {v['steps']}번) {v['check_notes']}")
    print(f"   {c['claim_ko']}")
    print(f"   {v['explanation_ko']}")
print(f"\n토큰 합계: in {total['input']:,} / out {total['output']:,}")

PRICE = {"gpt-4o": (2.50, 10.00), "gpt-4o-mini": (0.15, 0.60), "gpt-4.1-mini": (0.40, 1.60)}   # $ / 100만 토큰
p_in, p_out = PRICE.get(MODEL, (0, 0))
usd = total["input"] / 1e6 * p_in + total["output"] / 1e6 * p_out
print(f"비용: 약 ${usd:.3f} (약 {usd * 1400:,.0f}원, 1달러 1,400원 기준)")

## 7. 결과 확인

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
cols = ["claim_id", "speaker", "verdict", "steps", "claim_ko", "explanation_ko", "source_date", "check_notes"]
pd.set_option("display.max_colwidth", 90)
df[cols].set_index("claim_id")

In [ ]:
# 에이전트가 한 일 따라가 보기 (번호만 바꾸면 돼요)
def show_trace(i: int = 0):
    r = results[i]
    print(f"[{r['claim_id']}] {r['claim_ko']}\n")
    for t in r["trace"]:
        print(f"  {t['step']}단계 {t['tool']}({json.dumps(t['args'], ensure_ascii=False)[:120]})")
        print(f"         → {t['result'][:200]}...\n")
    print(f"  판정: {r['verdict']}  |  근거: {r['source_url']} ({r['source_date']})")
    print(f"  발췌: \"{r['evidence_quote'][:200]}\"")
    print(f"  설명: {r['explanation_ko']}")

show_trace(0)

## 8. 저장

In [ ]:
suffix = "" if (TEST_PROPOSALS is None and MAX_CLAIMS is None) else "_test"
(OUT_DIR / f"{FILING}_verdicts{suffix}.json").write_text(
    json.dumps({"ticker": TICKER, "as_of": AS_OF, "model": MODEL, "usage": total, "results": results},
               ensure_ascii=False, indent=2), encoding="utf-8")
flat = [{k: v for k, v in r.items() if k != "trace"} for r in results]
if flat:
    with (OUT_DIR / f"{FILING}_verdicts{suffix}.csv").open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=list(flat[0].keys()))
        w.writeheader(); w.writerows(flat)
print("저장 완료:", OUT_DIR)

## 다음 단계
7번 셀 결과와 `show_trace(0)` 출력을 보내 주세요. 같이 볼 것:
- 판정이 맞는지 (직접 근거를 열어서 확인)
- 에이전트가 **엉뚱한 곳을 헤매지 않았는지** (도구 횟수, 검색어)
- 검사관이 판정을 낮춘 경우가 있는지 (`check_notes`)